# Data preparation

In [1]:
import json
import random
from pathlib import Path
from utils import load_images

my_random = 2441156
metadata_path = '../../../../Datasets/PlantVillage-Dataset/leaf_grouping/leaf-map.json'

### Get images data
Use prepared utility function based on functions developed in 01_eda.ipynb

In [2]:
images_data = load_images()

images_path = images_data["images_path"]
classes = images_data["classes"]
number_classes = images_data["number_classes"]
class_counts = images_data["class_counts"]
class_distribution = images_data["class_distribution"]
clean_images = images_data["clean_images"]
all_rgb = images_data["all_rgb"]

print(f"Dataset Path: {images_path}")
print(f"Total Classes: {number_classes}")
print(f"Total Clean Images: {len(clean_images)}")
print(f"Are all images valid RGB JPEGs? {all_rgb}")
print(f"Smallest class size: {min(class_counts)}")
print(f"Largest class size: {max(class_counts)}")

Validating image properties (RGB / JPEG)...


100%|██████████| 54304/54304 [00:13<00:00, 4115.65it/s]

Dataset Path: C:\Users\kmiel\Repository\ARU MSc\Datasets\PlantVillage-Dataset\raw\color
Total Classes: 38
Total Clean Images: 54304
Are all images valid RGB JPEGs? True
Smallest class size: 152
Largest class size: 5507


### Import images metadata
Import metadata from leaf-map.json file in repository to inspect it

In [3]:
with open(metadata_path) as metadata_file:
    images_metadata = json.load(metadata_file)

print(f"1. Type: {type(images_metadata)}")

print(f"2. Total top-level entries: {len(images_metadata)}")

# Key and value types of one example entry
sample_key, sample_value = next(iter(images_metadata.items()))

print(f"3. Example key type: {type(sample_key)} (e.g., '{sample_key}')")
print(f"   Example value type: {type(sample_value)}")


1. Type: <class 'dict'>
2. Total top-level entries: 40328
3. Example key type: <class 'str'> (e.g., 'rutg._hl 3733')
   Example value type: <class 'list'>


#### Investigate metadata

- How many keys have exactly one value?
- How many keys have more than one value?
- What is the maximum number of values associated with a single key?

In [4]:
# Number of keys with one value
one_value_count = sum(1 for values in images_metadata.values() if len(values) == 1)

# Number of keys with multiple values
multiple_values_count = sum(1 for values in images_metadata.values() if len(values) > 1)

# Max number of values per key
max_values = max(len(value) for value in images_metadata.values())

print(f"Keys with exactly one value: {one_value_count}")
print(f"Keys with more than one value: {multiple_values_count}")
print(f"Maximum number of values for a single key: {max_values}")

Keys with exactly one value: 38156
Keys with more than one value: 2172
Maximum number of values for a single key: 3


In [18]:
total_individual_mappings = sum(len(values) for values in images_metadata.values())
print(f'Total number of individual mappings: {total_individual_mappings}')

Total number of individual mappings: 42520


#### Extract the image identifier
Image file names follow this pattern: `0b8dabb7-5f1b-4fdc-b3fa-30b289707b90___JR_FrgE.S 3047.JPG` while the metadata resembles `jr_frge.s 3047`.  
The identifier from the image files must be extracted in order to lookup the metadata in the json file.

In [85]:
# helper function to extract image identifier names
def get_image_id(image_path: Path):
    # remove the file extension
    stem = image_path.stem
    # split by '___' if exists in name, if not return the stem
    if "___" in stem:
        identifier = stem.split("___")[1]
    else:
        identifier = stem

    # standardise to lower case as this is what leaf-map.json uses
    return identifier.lower()

Test the get_image_id

In [97]:
test_paths = [
    Path("Soybean___healthy/637c4131___RS_HL 4655.JPG"),
    Path("Corn_(maize)___Common_rust_/RS_Rust 1563.JPG")
]

for path in test_paths:
    print(get_image_id(path))

rs_hl 4655
rs_rust 1563


Inspect how the directory, file and metadata naming relates to each other using 3 examples

In [86]:
def split_metadata(mapping):
    return mapping.split(":::")

## Identify all naming differences
The original repository metadata contains the class-specific Apple_Frogeye Spot.csv while the published 38-class image dataset uses Apple___Black_rot.
This code identifies the discrepancies.

In [87]:
directory_classes = set(classes)
metadata_classes = set()

for mapping_list in images_metadata.values():
    for item in mapping_list:
        # separate class and leaf-group
        leaf_class, _ = split_metadata(item)
        metadata_classes.add(leaf_class)

print("Only in metadata:", metadata_classes - directory_classes)
print("Only in directories:", directory_classes - metadata_classes)
    

Only in metadata: {'Apple_Frogeye Spot'}
Only in directories: {'Corn_(maize)___Common_rust_', 'Squash___Powdery_mildew', 'Apple___Black_rot', 'Grape___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___healthy', 'Tomato___Tomato_mosaic_virus', 'Corn_(maize)___Northern_Leaf_Blight'}


### Extract specific image ID from metadata mapping - 3-image sample 
Based on class and filename find mapping in the metadata leaf-map.json and extract its leaf ID

In [88]:
random.seed(my_random)

images_to_check = random.sample(clean_images, 3)

print(f'Number of images to check: {len(images_to_check)}\n')

for img_path in images_to_check:
    image_id = get_image_id(img_path)

    print(f'Filename: {img_path.name}')
    print(f'Image ID: {image_id}')

    # [] returns an empty list if identifier is missing (image identifier not found in metadata)
    mapping = images_metadata.get(image_id, [])

    print(f'Mapping: {mapping}')

    for item in mapping:
        leaf_class, leaf_id = split_metadata(item)
        print(f'Split mapping:\n    Leaf class: {leaf_class}\n    Leaf group: {leaf_id}')

    class_name = img_path.parent.name

    print(f'Class: {class_name}')
    print(f'Leaf ID: {leaf_id}\n')

Number of images to check: 3

Filename: 637c4131-9177-4e3c-88fa-7fcacbca33ba___RS_HL 4655.JPG
Image ID: rs_hl 4655
Mapping: ['Soybean___healthy:::218.0']
Split mapping:
    Leaf class: Soybean___healthy
    Leaf group: 218.0
Class: Soybean___healthy
Leaf ID: 218.0

Filename: 790cfb1d-4926-46bb-bd50-b750e823c72a___Crnl_L.Mold 7130.JPG
Image ID: crnl_l.mold 7130
Mapping: ['Tomato___Leaf_Mold:::121.0']
Split mapping:
    Leaf class: Tomato___Leaf_Mold
    Leaf group: 121.0
Class: Tomato___Leaf_Mold
Leaf ID: 121.0

Filename: 61a05330-1086-4891-8482-3f422ff59492___GCREC_Bact.Sp 3429.JPG
Image ID: gcrec_bact.sp 3429
Mapping: ['Tomato___Bacterial_spot:::121.0']
Split mapping:
    Leaf class: Tomato___Bacterial_spot
    Leaf group: 121.0
Class: Tomato___Bacterial_spot
Leaf ID: 121.0



### Define function to make the image ID lookup reusable accross the dataset

In [89]:
def get_leaf_id(image_path: Path, images_metadata: dict):

    image_id = get_image_id(image_path)
    class_name = image_path.parent.name
    mapping = images_metadata.get(image_id, [])

    for item in mapping:
      leaf_class, leaf_id = split_metadata(item)

      if leaf_class == class_name:
          return leaf_id
    
    return None

#### Test the function

On 3 test images

In [90]:
random.seed(my_random)

images_to_check = random.sample(clean_images, 3)

print(f'Number of images to check: {len(images_to_check)}\n')

for img_path in images_to_check:
    leaf_id = get_leaf_id(img_path, images_metadata)

    print(f'Image Id: {leaf_id}')

Number of images to check: 3

Image Id: 218.0
Image Id: 121.0
Image Id: 121.0


On image identifier belonging to different class:

In [93]:
# RS_HL 4655 is Soybean___healthy
test_path = Path("Tomato___healthy/12345678___RS_HL 4655.JPG")

leaf_id = get_leaf_id(test_path, images_metadata)

print(f"Leaf ID: {leaf_id}")

Leaf ID: None


On image identifier not existing in metadata.

In [94]:
test_path = Path("Tomato___healthy/12345678___unknown_image.JPG")

leaf_id = get_leaf_id(test_path, images_metadata)

print(f"Leaf ID: {leaf_id}")

Leaf ID: None


In [95]:
unexpected_filenames = []

for img_path in clean_images:
    if "___" not in img_path.stem:
        unexpected_filenames.append(img_path)

print(f"Unexpected filenames: {len(unexpected_filenames)}")

for img_path in unexpected_filenames[:10]:
    print(img_path.parent.name, "→", img_path.name)

Unexpected filenames: 1192
Corn_(maize)___Common_rust_ → RS_Rust 1563.JPG
Corn_(maize)___Common_rust_ → RS_Rust 1564.JPG
Corn_(maize)___Common_rust_ → RS_Rust 1565.JPG
Corn_(maize)___Common_rust_ → RS_Rust 1566.JPG
Corn_(maize)___Common_rust_ → RS_Rust 1567.JPG
Corn_(maize)___Common_rust_ → RS_Rust 1568.JPG
Corn_(maize)___Common_rust_ → RS_Rust 1569.JPG
Corn_(maize)___Common_rust_ → RS_Rust 1570.JPG
Corn_(maize)___Common_rust_ → RS_Rust 1571.JPG
Corn_(maize)___Common_rust_ → RS_Rust 1572.JPG


### Now iterate over the clean images to calculate how many images have valid leaf ID in the metadata file

In [98]:
matched_count = 0
unresolved_count = 0

for img_path in clean_images:
    leaf_id = get_leaf_id(img_path, images_metadata)

    if leaf_id is not None:
        matched_count += 1
    else:
        unresolved_count += 1

print(f'Matched leaf IDs: {matched_count}')
print(f'Unresolved leaf IDs: {unresolved_count}')
print(f'Metadata coverage: {(matched_count/len(clean_images)) * 100} %')

Matched leaf IDs: 40490
Unresolved leaf IDs: 13814
Metadata coverage: 74.56172657631114 %


#### Count unresolved images in each class

In [112]:
from collections import Counter

unresolved_by_class = Counter()

for img_path in clean_images:
    leaf_id = get_leaf_id(img_path, images_metadata)

    if leaf_id is None:
        class_name = img_path.parent.name

        unresolved_by_class[class_name] += 1

for class_name, count in unresolved_by_class.most_common():
    print(f'  {class_name}: {count}')

  Tomato___Tomato_Yellow_Leaf_Curl_Virus: 2619
  Squash___Powdery_mildew: 1835
  Tomato___Target_Spot: 1404
  Corn_(maize)___Common_rust_: 1192
  Corn_(maize)___healthy: 1162
  Tomato___Late_blight: 989
  Corn_(maize)___Northern_Leaf_Blight: 985
  Tomato___Septoria_leaf_spot: 771
  Apple___Black_rot: 621
  Tomato___healthy: 592
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 513
  Grape___healthy: 423
  Tomato___Tomato_mosaic_virus: 373
  Strawberry___Leaf_scorch: 333
  Pepper,_bell___healthy: 1
  Tomato___Early_blight: 1


Above reveals that the unresolved images are distributes accross 16 classes.  
Two causes are possible:
- Missing data in leaf-map.json
- image identifiers do not match directory in metadata mappings

Investigate how many examples cover each case:

In [117]:
missing_identifier = Counter()
class_mismatch = Counter()

for img_path in clean_images:
    image_id = get_image_id(img_path)
    class_name = img_path.parent.name
    
    if image_id not in images_metadata:
        missing_identifier[class_name] += 1
    else:
        leaf_id = get_leaf_id(img_path, images_metadata)
        if leaf_id is None:
            class_mismatch[class_name] += 1

print('Missing Identifiers in Metadata Count:')
for class_name, count in missing_identifier.most_common():
    print(f"  {class_name}: {count}")

print('\nClass Mismatches (ID found, but no mapping) Count:')
for class_name, count in class_mismatch.most_common():
    print(f"  {class_name}: {count}")

print(f'\nTotal {missing_identifier.total() + class_mismatch.total()}')

Missing Identifiers in Metadata Count:
  Tomato___Tomato_Yellow_Leaf_Curl_Virus: 2619
  Squash___Powdery_mildew: 1835
  Tomato___Target_Spot: 1404
  Corn_(maize)___Common_rust_: 1192
  Corn_(maize)___healthy: 1162
  Tomato___Late_blight: 989
  Corn_(maize)___Northern_Leaf_Blight: 985
  Tomato___Septoria_leaf_spot: 771
  Tomato___healthy: 592
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 513
  Grape___healthy: 423
  Tomato___Tomato_mosaic_virus: 373
  Strawberry___Leaf_scorch: 333
  Pepper,_bell___healthy: 1
  Tomato___Early_blight: 1

Class Mismatches (ID found, but no mapping) Count:
  Apple___Black_rot: 621

Total 13814


Apple___Black_rot is the only class with ID in no mapping class.  
The dataset directory suggests that it is mapped to Apple_Frogeye images as they appear with this name in Apple___Black_rot directory.  
The rest of the images have no information in leaf-map.json at all.

Inspect one image from Apple___Black_rot directory and its metadata entry to confirm the suspicion:

In [118]:
apple_images = [
    img_path for img_path in clean_images
    if img_path.parent.name == "Apple___Black_rot"
]

sample_apple = apple_images[0]

image_id = get_image_id(sample_apple)
mapping = images_metadata.get(image_id)

print(f"Filename: {sample_apple.name}")
print(f"Image ID: {image_id}")
print(f"Metadata: {mapping}")

Filename: 0090d05d-d797-4c99-abd4-3b9cb323a5fd___JR_FrgE.S 8727.JPG
Image ID: jr_frge.s 8727
Metadata: ['Apple_Frogeye Spot:::56.0']
